# 🌿 AgriVision: Plant Disease Detection Master Pipeline

This notebook consolidates the entire AgriVision pipeline into a single interactive environment. 
It covers everything from data ingestion via TFDS to Deep Learning classification and Explainable AI.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
import joblib
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

print("✅ Environment Setup Complete")
print(f"TensorFlow version: {tf.__version__}")

## 1. Preprocessing & Segmentation Logic

In [ ]:
def preprocess_image(image_bgr):
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    return {"rgb": rgb, "gray": gray, "hsv": hsv, "blurred": blurred}

def segment_leaf_hsv(hsv_image):
    # Lower and upper bounds for green/yellow (leaf colors)
    lower_green = np.array([25, 40, 40])
    upper_green = np.array([90, 255, 255])
    mask = cv2.inRange(hsv_image, lower_green, upper_green)
    return mask

def detect_edges(gray_image):
    canny = cv2.Canny(gray_image, 100, 200)
    sobelx = cv2.Sobel(gray_image, cv2.CV_64F, 1, 0, ksize=5)
    sobely = cv2.Sobel(gray_image, cv2.CV_64F, 0, 1, ksize=5)
    sobel = cv2.magnitude(sobelx, sobely)
    return {"canny": canny, "sobel": np.uint8(sobel)}

def kmeans_segmentation(rgb_image, k=3):
    z = rgb_image.reshape((-1, 3))
    z = np.float32(z)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
    _, label, center = cv2.kmeans(z, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    center = np.uint8(center)
    res = center[label.flatten()]
    return res.reshape((rgb_image.shape))

## 2. Feature Extraction (Classical ML)

In [ ]:
def extract_feature_vector(rgb, hsv, gray, mask):
    # 1. Color Histogram Features
    hist_h = cv2.calcHist([hsv], [0], mask, [16], [0, 180]).flatten()
    hist_s = cv2.calcHist([hsv], [1], mask, [16], [0, 256]).flatten()
    
    # 2. Shape/Area features
    leaf_area = np.sum(mask > 0)
    total_area = mask.shape[0] * mask.shape[1]
    area_ratio = leaf_area / total_area
    
    # 3. Simple Texture (Standard Deviation of Grayscale)
    std_dev = np.std(gray[mask > 0]) if leaf_area > 0 else 0
    
    return np.concatenate([hist_h, hist_s, [area_ratio, std_dev]])

## 3. TFDS Data Pipeline

In [ ]:
def load_agrivision_dataset(batch_size=32, image_size=(224, 224)):
    dataset, info = tfds.load(
        name="plant_village",
        split="train",
        with_info=True,
        as_supervised=True,
        shuffle_files=True
    )
    
    def preprocess(image, label):
        image = tf.image.resize(image, image_size)
        # Important: Pass as uint8 [0, 255] for Keras preprocess_input layer
        image = tf.cast(image, tf.float32)
        return image, label

    dataset = dataset.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    return dataset, info

dataset, info = load_agrivision_dataset()
class_names = info.features['label'].names
print(f"Loaded {len(class_names)} classes.")

## 4. Deep Learning Training (MobileNetV2)

In [ ]:
def build_dl_model(num_classes):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet",
    )
    base.trainable = False
    base._name = "mobilenet_v2_base"

    inp = tf.keras.layers.Input(shape=(224, 224, 3))
    # Standard Keras preprocessing
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inp)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    out = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    
    model = tf.keras.models.Model(inp, out)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

dl_model = build_dl_model(len(class_names))
print("🤖 Deep Learning Model Architecture Ready")

## 5. Master Execution Phase

We will now extract features for Classical ML and then train the Deep Learning model.

In [ ]:
# Phase 1: ML Feature Extraction (using 2048 samples for speed)
print("🔍 Extracting ML Features...")
X, y = [], []
for batch_images, batch_labels in tqdm(dataset.take(64), desc="ML Processing"): # ~2048 samples
    for img_tf, label_idx in zip(batch_images.numpy(), batch_labels.numpy()):
        img_bgr = cv2.cvtColor(img_tf.astype(np.uint8), cv2.COLOR_RGB2BGR)
        pp = preprocess_image(img_bgr)
        mask = segment_leaf_hsv(pp["hsv"])
        feat = extract_feature_vector(pp["rgb"], pp["hsv"], pp["gray"], mask)
        X.append(feat)
        y.append(class_names[label_idx])

X = np.array(X)
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Phase 2: Train Classical ML (Random Forest)
print("🧠 Training Random Forest...")
rf = Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=100))])
rf.fit(X, y_enc)
print(f"✅ ML Model Accuracy (Train): {rf.score(X, y_enc):.2%}")

# Phase 3: Train Deep Learning (1 epoch for demo)
print("🤖 Training MobileNetV2...")
dl_model.fit(dataset.take(100), epochs=1) # Reduced for demo
print("✅ DL Model Pass Complete")

## 6. Inference Test

Pick a random image from the dataset and see what both models think!

In [ ]:
for images, labels in dataset.take(1):
    test_img = images[0].numpy().astype(np.uint8)
    true_label = class_names[labels[0]]
    
    # DL Prediction
    dl_pred = dl_model.predict(np.expand_dims(test_img, 0), verbose=0)
    dl_idx = np.argmax(dl_pred)
    dl_conf = dl_pred[0][dl_idx]
    
    # ML Prediction
    img_bgr = cv2.cvtColor(test_img, cv2.COLOR_RGB2BGR)
    pp = preprocess_image(img_bgr)
    mask = segment_leaf_hsv(pp["hsv"])
    ml_feat = extract_feature_vector(pp["rgb"], pp["hsv"], pp["gray"], mask).reshape(1, -1)
    ml_idx = rf.predict(ml_feat)[0]
    
    plt.imshow(test_img)
    plt.title(f"True: {true_label}\nDL: {class_names[dl_idx]} ({dl_conf:.1%})\nML: {le.classes_[ml_idx]}")
    plt.axis('off')
    plt.show()